In [ ]:
import random
import csv
import heapq
import math

# Define the JobSeeker and JobOffer classes
class JobSeeker:
    def __init__(self, skills, experience, salary, location, job_interest, sector, education_level, job_id):
        self.skills = skills
        self.experience = experience
        self.salary = salary
        self.location = location
        self.job_interest = job_interest
        self.sector = sector
        self.education_level = education_level
        self.job_id = job_id
    
    def __repr__(self):
        return f"JobSeeker({self.job_id})"

class JobOffer:
    def __init__(self, required_skills, min_experience, salary_range, location, sector, education_level):
        self.required_skills = required_skills
        self.min_experience = min_experience
        self.salary_range = salary_range
        self.location = location
        self.sector = sector
        self.education_level = education_level
    
    def __repr__(self):
        return f"JobOffer({self.required_skills})"

# Heuristic function to evaluate how well a job seeker matches a job offer
def improved_heuristic(job_seeker, job_offer):
    score = 0
    
    # Skill match: Add points for each matching skill
    job_seeker_skills = job_seeker.skills.lower().split(", ") if isinstance(job_seeker.skills, str) else [s.lower() for s in job_seeker.skills]
    required_skills = [skill.lower() for skill in job_offer.required_skills]
    
    skill_match = len(set(job_seeker_skills) & set(required_skills))
    score += skill_match * 2  # Double points for skill matches
    
    # Experience match: Deduct points if the experience is less than required
    experience_gap = job_seeker.experience - job_offer.min_experience
    if experience_gap < 0:
        score += experience_gap * 2  # Penalty for being underqualified
    else:
        score += min(3, experience_gap)  # Cap at 3 points for excess experience
    
    # Salary match: Deduct points if salary is outside the offer range
    min_salary, max_salary = job_offer.salary_range
    if job_seeker.salary < min_salary:
        score -= min(3, (min_salary - job_seeker.salary) / min_salary * 10)
    elif job_seeker.salary > max_salary:
        score -= min(8, (job_seeker.salary - max_salary) / max_salary * 15)
    else:
        score += 2
    
    # Location match: Add points if location matches
    if job_seeker.location.lower() == job_offer.location.lower():
        score += 3
    
    # Sector match: Add points if sector matches
    if job_seeker.sector.lower() == job_offer.sector.lower():
        score += 2
    
    # Education match: Add points if education level matches or exceeds the job's requirements
    education_levels = ['high school', 'associate', 'bachelor', 'master', 'phd', 'doctorate']
    try:
        seeker_edu = next((i for i, level in enumerate(education_levels) if level in job_seeker.education_level.lower()), -1)
        required_edu = next((i for i, level in enumerate(education_levels) if level in job_offer.education_level.lower()), -1)
        
        if seeker_edu >= required_edu and required_edu >= 0:
            score += 2  # Meets or exceeds required education
        elif seeker_edu >= 0 and required_edu >= 0:
            score -= (required_edu - seeker_edu)  # Penalty based on education gap , max difference is 6 
    except AttributeError:
        # Handle case where education_level is not accessible as expected
        pass
    
    return score

# Find top 5 matches from original dataset
def find_top_matches(job_seekers, job_offer, top_n=5):
    matches = []
    counter = 0  # Add a counter for tiebreaking
    
    for job_seeker in job_seekers:
        fitness = improved_heuristic(job_seeker, job_offer)
        counter += 1  # Increment counter
        heapq.heappush(matches, (-fitness, counter, job_seeker))  # Add counter as tiebreaker
        
        # Keep only top N
        if len(matches) > top_n:
            heapq.heappop(matches)
        
    # Return in descending order by fitness
    results = []
    while matches:
        fitness, _, seeker = heapq.heappop(matches)
        results.insert(0, (seeker, -fitness))  # Negate back to positive
        
    return results


# Genetic Algorithm for job matching
class GeneticAlgorithm:
    def __init__(self, job_seekers, job_offer, generations=50, mutation_rate=0.1):
        self.job_seekers = job_seekers
        self.job_offer = job_offer
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.original_job_seekers = {seeker.job_id: seeker for seeker in job_seekers if hasattr(seeker, 'job_id')}
        self.population_size = self.estimate_population_size()
    
    def estimate_population_size(self):
        confidence_level=0.95 
        error_margin=0.05
        sigma=0.5
        Z=1.96
        #this will calculate the population size (sample) based on z-score and the size of the dataSet and the margin of eror alpha=0.05
        #we prooved that n<N in the problem formulation
        n = (Z * sigma / error_margin) ** 2
        return round(n)

    def create_population(self):
        # Create an initial population by randomly selecting job seekers
        return random.sample(self.job_seekers, self.population_size)

    def fitness(self, job_seeker):
        # Evaluate the fitness based on how well the job seeker matches the job offer
        return improved_heuristic(job_seeker, self.job_offer)

    def select_parents(self, population):

        tournament_size = min(10, int(len(population) * 0.02)) 
        
        # Cache fitness for efficiency
        fitness_cache = {indiv: self.fitness(indiv) for indiv in population}
        
        def select_one(exclude=None):
            candidates = random.sample(population, tournament_size)
            if exclude and exclude in candidates:
                candidates.remove(exclude)  # Avoid selecting the same parent twice
            
            fitnesses = [fitness_cache[c] for c in candidates]
            
            # Shift all fitnesses to be positive , because we cant have negative probability 
            min_fitness = min(fitnesses)
            if min_fitness < 0:
                shifted_fitnesses = [f + abs(min_fitness)  for f in fitnesses]  
            else:
                shifted_fitnesses = fitnesses  # No shift needed if all fitnesses ≥ 0
            
            total_fitness = sum(shifted_fitnesses)
            probabilities = [f / total_fitness for f in shifted_fitnesses]
            
            return random.choices(candidates, weights=probabilities, k=1)[0]

        parent1 = select_one()
        parent2 = select_one(exclude=parent1)
        return parent1, parent2


    def crossover(self, parent1, parent2):
        # Combine the parents' attributes to create an offspring
        child_skills = random.choice([parent1.skills, parent2.skills])
        child_experience = random.choice([parent1.experience, parent2.experience])
        child_salary = random.choice([parent1.salary, parent2.salary])
        child_location = random.choice([parent1.location, parent2.location])
        child_sector = random.choice([parent1.sector, parent2.sector])
        child_education_level = random.choice([parent1.education_level, parent2.education_level])
        
        # Generate a new unique ID (negative to differentiate from real job seekers)
        child_id = -random.randint(1, 1000000)

        return JobSeeker(child_skills, child_experience, child_salary, child_location, 
                        parent1.job_interest, child_sector, child_education_level, child_id)

    def mutate(self, job_seeker):
        # Mutate some of the attributes randomly
        if random.random() < self.mutation_rate:
            job_seeker.skills = random.choice(self.job_seekers).skills
        if random.random() < self.mutation_rate:
            job_seeker.experience = random.randint(0, 10)
        if random.random() < self.mutation_rate:
            job_seeker.salary = random.randint(30000, 100000)
        if random.random() < self.mutation_rate:
            job_seeker.location = random.choice([s.location for s in random.sample(self.job_seekers, min(5, len(self.job_seekers)))])
        if random.random() < self.mutation_rate:
            job_seeker.sector = random.choice([s.sector for s in random.sample(self.job_seekers, min(5, len(self.job_seekers)))])
        if random.random() < self.mutation_rate:
            job_seeker.education_level = random.choice([s.education_level for s in random.sample(self.job_seekers, min(5, len(self.job_seekers)))])

    def evolve(self):
        population = self.create_population()
        best_overall = None
        best_fitness = float('-inf')
        
        for generation in range(self.generations):
            if generation % 10 == 0:
                print(f"Generation {generation + 1}")
            
            # Evaluate the fitness of each individual and sort population (best first)
            population.sort(key=self.fitness, reverse=True)
            
            # Track the best individual so far
            current_best = population[0]
            current_fitness = self.fitness(current_best)
            if current_fitness > best_fitness:
                best_fitness = current_fitness
                best_overall = current_best

            offspring =[]
            # Create offspring until we have at least self.population_size individuals in offspring
            while len(offspring) < self.population_size:
                parent1, parent2 = self.select_parents(population)
                child = self.crossover(parent1, parent2)
                self.mutate(child)
                offspring.append(child)
            
            # Combine the original population with the new offspring
            combined_population = population + offspring
            
            # Sort the combined population based on fitness in descending order
            combined_population.sort(key=self.fitness, reverse=True)
            
            # Select the best k individuals, where k equals the desired population size
            population = combined_population[:self.population_size]
        
        # After the generations, sort and pick the top 5 solutions
        ga_top_5 = population[:5]
        
        # Map evolved solutions to the closest match in the original dataset:
        mapped_solutions = []
        for evolved_seeker in ga_top_5:
            best_original = None
            best_similarity = float('-inf')
            evolved_fitness = self.fitness(evolved_seeker)
            for original_seeker in self.job_seekers:
                if hasattr(original_seeker, 'job_id') and original_seeker.job_id > 0:  # Only consider real job seekers
                    original_fitness = self.fitness(original_seeker)
                    similarity = original_fitness  # Fitness used as similarity measure
                    if similarity > best_similarity:
                        best_similarity = similarity
                        best_original = original_seeker
            if best_original:
                mapped_solutions.append((evolved_seeker, evolved_fitness, best_original, best_similarity))
        
        return mapped_solutions

# Load job seekers from CSV
def load_job_seekers(filename):
    job_seekers = []
    try:
        with open(filename, newline='', encoding='utf-8') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                job_seekers.append(JobSeeker(
                    skills=row['skills'],
                    experience=int(row['experience']),
                    salary=int(row['salary']),
                    location=row['location'],
                    job_interest=row['job_interest'],
                    sector=row['sector'],
                    education_level=row['education_level'],
                    job_id=int(row['job_id'])
                ))
        print(f"Loaded {len(job_seekers)} job seekers.")
    except FileNotFoundError:
        print(f"Error: The file {filename} was not found.")
    except Exception as e:
        print(f"Error: {e}")
    return job_seekers

# Define some sample job offers for testing
def define_sample_jobs():
    return [
        JobOffer(
            required_skills=["SEO", "Google Ads"],
            min_experience=3,
            salary_range=(50000, 90000),
            location="Algiers",
            sector="Business",
            education_level="Master's"
        ),
        JobOffer(
            required_skills=["Java", "SQL"],
            min_experience=2,
            salary_range=(40000, 80000),
            location="Tizi Ouzou",
            sector="Information Technology",
            education_level="Bachelor's"
        )
    ]

def main():
    job_seekers = load_job_seekers("job_seekers_with_ids.csv")
    
    if not job_seekers:
        print("No job seekers found. Check the file.")
        return
    
    job_offers = define_sample_jobs()

    for i, job_offer in enumerate(job_offers):
        print(f"\n{'='*80}")
        print(f"JOB OFFER {i+1}: {job_offer}")
        print(f"{'='*80}")
        
        # First, get the top 5 matches directly from the dataset
        print("\nTOP 5 MATCHES FROM ORIGINAL DATASET:")
        print(f"{'ID':^5} | {'Fitness':^10} | {'Skills':^30} | {'Exp':^5} | {'Salary':^10} | {'Location':^15} | {'Sector':^10}")
        print("-" * 100)
        
        top_matches = find_top_matches(job_seekers, job_offer, top_n=5)
        for seeker, fitness in top_matches:
            skills_str = seeker.skills[:30] + "..." if len(str(seeker.skills)) > 30 else seeker.skills
            print(f"{seeker.job_id:^5} | {fitness:^10.2f} | {skills_str:^30} | {seeker.experience:^5} | {seeker.salary:^10} | {seeker.location:^15} | {seeker.sector:^10}")
        
        print("\nRUNNING GENETIC ALGORITHM OPTIMIZATION...")
        ga = GeneticAlgorithm(job_seekers, job_offer, generations=50, mutation_rate=0.1)
        mapped_solutions = ga.evolve()
        
        print("\nTOP 5 MATCHES FROM GENETIC ALGORITHM WITH MAPPINGS:")
        print(f"{'#':^3} | {'GA Fitness':^10} | {'Best Match ID':^12} | {'Original Fitness':^15} | {'Skills':^20}")
        print("-" * 80)
        
        for i, (evolved, evolved_fitness, original, original_fitness) in enumerate(mapped_solutions):
            skills_str = str(evolved.skills)[:20] + "..." if len(str(evolved.skills)) > 20 else str(evolved.skills)
            print(f"{i+1:^3} | {evolved_fitness:^10.2f} | {original.job_id:^12} | {original_fitness:^15.2f} | {skills_str:^20}")

if __name__ == "__main__":
    main()